# Bài học 03 - Các Mẫu Thiết Kế Agentic

Trong bài học này, chúng ta khám phá ba mẫu thiết kế nền tảng để xây dựng các agent AI hiệu quả:

1. **Hướng Dẫn Rõ Ràng Cho Agent** — Tạo ra các lời nhắc chính xác, xác định vai trò để hướng dẫn hành vi của agent
2. **Đầu Ra Có Cấu Trúc với Mô Hình Pydantic** — Đảm bảo các agent trả về dữ liệu dự đoán được và đã được xác thực
3. **Agent Đơn Trách Nhiệm** — Thiết kế các agent tập trung, mỗi agent thực hiện tốt một việc

Chúng ta sẽ áp dụng từng mẫu cho một kịch bản **gợi ý điểm đến du lịch**, từng bước xây dựng một hệ thống có thể đề xuất điểm đến, kiểm tra tình trạng sẵn có và xử lý logistics.


## Cài đặt


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic python-dotenv --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Mẫu 1: Hướng dẫn rõ ràng cho Đại lý

Mẫu có ảnh hưởng lớn nhất cũng là mẫu đơn giản nhất: viết hướng dẫn rõ ràng, chi tiết cho đại lý của bạn.

Hướng dẫn tốt xác định:
- **Ai** là đại lý (nhân vật và giọng điệu)
- **Cái gì** nó nên làm (trách nhiệm từng bước)
- **Cách** nó nên hành xử (hạn chế và phong cách)

Dưới đây, chúng tôi tạo một đại lý trợ lý du lịch với hướng dẫn rõ ràng định hình mọi phản hồi nó tạo ra.


In [ ]:
agent = provider.as_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Mẫu 2: Đầu ra có cấu trúc với các mô hình Pydantic

Văn bản tự do hữu ích cho hội thoại, nhưng các hệ thống hạ nguồn cần dữ liệu có cấu trúc.
Bằng cách kết hợp **các mô hình Pydantic** với một **hàm công cụ**, chúng ta có thể:

- Định nghĩa một schema chính xác cho đầu ra của tác nhân
- Tự động xác thực các phản hồi
- Tích hợp kết quả của tác nhân vào logic ứng dụng một cách đáng tin cậy

Chìa khóa để thi hành là truyền `response_format` khi chúng ta chạy tác nhân. Điều này bắt buộc
mô hình trả về một đối tượng `TravelRecommendations` đã được xác thực (có trên `response.value`)
thay vì văn bản tự do. Công cụ `get_destination_details` cũng trả về một kiểu
`DestinationRecommendation`, vì vậy dữ liệu giữ được cấu trúc xuyên suốt.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(
    destination: Annotated[str, "The destination to look up"]
) -> DestinationRecommendation:
    """Get structured details about a vacation destination."""
    details = {
        "Barcelona": DestinationRecommendation(
            destination="Barcelona",
            available=True,
            best_season="May-Jun",
            highlights=["Beach", "Architecture", "Nightlife"],
            estimated_budget_usd=2000,
        ),
        "Tokyo": DestinationRecommendation(
            destination="Tokyo",
            available=True,
            best_season="Mar-Apr",
            highlights=["Culture", "Food", "Technology"],
            estimated_budget_usd=2500,
        ),
        "Cape Town": DestinationRecommendation(
            destination="Cape Town",
            available=False,
            best_season="Nov-Mar",
            highlights=["Nature", "Wine", "Adventure"],
            estimated_budget_usd=1800,
        ),
    }
    return details.get(
        destination,
        DestinationRecommendation(
            destination=destination,
            available=False,
            best_season="Unknown",
            highlights=[],
            estimated_budget_usd=0,
        ),
    )


structured_agent = provider.as_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

# Passing `response_format` forces the agent to return a validated
# TravelRecommendations object instead of free-form text.
response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget",
    options={"response_format": TravelRecommendations},
)

if response and response.value:
    result: TravelRecommendations = response.value
    for rec in result.recommendations:
        status = "Available" if rec.available else "Not available"
        print(f"{rec.destination} ({status})")
        print(f"  Best season: {rec.best_season}")
        print(f"  Highlights: {', '.join(rec.highlights)}")
        print(f"  Estimated budget: ${rec.estimated_budget_usd}")
        print()
    print(f"Note: {result.personalized_note}")
else:
    print("No validated structured response was returned.")
    print(response)


## Mẫu 3: Các Đại lý Đơn trách nhiệm

Các nhiệm vụ phức tạp được hưởng lợi từ việc phân chia công việc cho nhiều đại lý tập trung, mỗi đại lý có một trách nhiệm duy nhất:

- Một **Chuyên gia Điểm đến** biết về địa điểm và tính khả dụng
- Một **Người lập kế hoạch Hậu cần** xử lý các chuyến bay, khách sạn và hành trình

Điều này phản ánh nguyên tắc kỹ thuật phần mềm về *tách biệt các mối quan tâm* — mỗi đại lý sẽ dễ dàng kiểm thử, bảo trì và cải tiến một cách độc lập hơn.


In [ ]:
destination_agent = provider.as_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = provider.as_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Tóm tắt

Trong bài học này, chúng ta đã áp dụng ba mẫu thiết kế theo hướng tác nhân vào một kịch bản đề xuất du lịch:

| Mẫu | Ý tưởng chính | Lợi ích |
|---|---|---|
| **Hướng dẫn rõ ràng** | Xác định persona, trách nhiệm và ràng buộc ngay từ đầu | Hành vi tác nhân nhất quán, phù hợp thương hiệu |
| **Đầu ra có cấu trúc** | Sử dụng các mô hình Pydantic làm định dạng phản hồi | Kết quả được xác thực, có thể đọc bởi máy |
| **Trách nhiệm đơn lẻ** | Giao cho mỗi tác nhân một công việc tập trung | Dễ kiểm thử, bảo trì và kết hợp |

Các mẫu này kết hợp một cách tự nhiên — bạn có thể kết hợp hướng dẫn rõ ràng với đầu ra có cấu trúc trong một tác nhân có trách nhiệm đơn lẻ để xây dựng các hệ thống chắc chắn, sẵn sàng cho sản xuất.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Tuyên bố miễn trừ trách nhiệm**:
Tài liệu này đã được dịch bằng dịch vụ dịch thuật AI [Co-op Translator](https://github.com/Azure/co-op-translator). Mặc dù chúng tôi cố gắng đảm bảo độ chính xác, xin lưu ý rằng bản dịch tự động có thể chứa lỗi hoặc sai sót. Tài liệu gốc bằng ngôn ngữ gốc nên được coi là nguồn tin chính thức. Đối với thông tin quan trọng, nên sử dụng dịch vụ dịch thuật chuyên nghiệp bởi con người. Chúng tôi không chịu trách nhiệm về bất kỳ hiểu lầm hoặc giải thích sai nào phát sinh từ việc sử dụng bản dịch này.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
